In [ ]:
# Install required libraries
!pip install -q langchain langchain-community langchain-core
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q pinecone
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.0 MB/s eta 0:00:00


In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
import os
from pypdf import PdfReader

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Pinecone

# Pinecone
import pinecone

# Groq
from groq import Groq

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving LLM_RAG_100_pages.pdf to LLM_RAG_100_pages.pdf


In [ ]:
# Get file name
pdf_file = list(uploaded.keys())[0]

reader = PdfReader(pdf_file)

text = ""
for page in reader.pages:
    text += page.extract_text()

print(text[:1000])  # preview

Page 1
Large Language Models (LLMs) are advanced artificial intelligence systems designed to
understand, process, and generate human-like language. They are built using deep learning
techniques, particularly transformer architectures, and are trained on massive datasets consisting
of text from books, websites, articles, and other sources. The goal of an LLM is to learn patterns,
relationships, grammar, semantics, and contextual meaning within language. LLMs function by
predicting the next word in a sequence based on the context of previous words. Through this
process, they can generate coherent sentences, answer questions, summarize information, and
assist in a wide range of language-related tasks. They rely on mathematical representations of text
called embeddings, which allow them to interpret meaning and similarity between words and
sentences. These models are pre-trained on large-scale datasets and can be further fine-tuned or
guided using prompts. Their capabilities include natura

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # max size of each chunk
    chunk_overlap=200,     # overlap between chunks
    length_function=len
)

# Split text into chunks
chunks = text_splitter.split_text(text)

# Preview
print("Total Chunks:", len(chunks))
print("\nSample Chunk:\n")
print(chunks[2])

Total Chunks: 300

Sample Chunk:

an LLM is to learn patterns, relationships, grammar, semantics, and contextual meaning within
language. LLMs function by predicting the next word in a sequence based on the context of
previous words. Through this process, they can generate coherent sentences, answer questions,
summarize information, and assist in a wide range of language-related tasks. They rely on
mathematical representations of text called embeddings, which allow them to interpret meaning and
similarity between words and sentences. These models are pre-trained on large-scale datasets and
can be further fine-tuned or guided using prompts. Their capabilities include natural language
understanding, reasoning, translation, and even code generation. Examples of LLMs include
models like GPT, LLaMA, and others developed by major AI organizations.
Page 2
Retrieval-Augmented Generation (RAG) is an advanced AI framework that enhances the


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/e5-large-v2"
)

# Convert chunks into embeddings
embeddings = embedding_model.embed_documents(chunks)

# Preview
print("Total Embeddings:", len(embeddings))
print("Embedding Vector Size:", len(embeddings[0]))

/tmp/ipykernel_4388/2119168953.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Total Embeddings: 300
Embedding Vector Size: 1024


In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone
pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

index_name = "rag--project"

# Create index (only if not exists)
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=len(embeddings[0]),  # embedding size
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1' # change if your region is different
        )
    )

# Connect to index
index = pc.Index(index_name)

# Prepare data for upload
vectors = []
for i, chunk in enumerate(chunks):
    vectors.append((
        str(i),                 # unique ID
        embeddings[i],          # vector
        {"text": chunk}         # metadata (VERY IMPORTANT)
    ))

# Upload to Pinecone
index.upsert(vectors=vectors)

print("✅ Data stored in Pinecone successfully!")

✅ Data stored in Pinecone successfully!


In [ ]:
# Convert query into embedding
def retrieve_docs(query, top_k=3):
    query_embedding = embedding_model.embed_query("query: " + query)

    # Search in Pinecone
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    # Extract text
    retrieved_chunks = [match["metadata"]["text"] for match in results["matches"]]

    return retrieved_chunks

# Test retrieval
query = "What is this document about?"
docs = retrieve_docs(query)

print("Retrieved Chunks:\n")
for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}:\n{doc[:300]}")

Retrieved Chunks:


Chunk 1:
source and use it to generate more accurate and context-aware responses. In a RAG system, when
a user submits a query, the system first converts the query into a vector representation using
embeddings. This vector is then used to search a vector database containing pre-processed
document chunks. The

Chunk 2:
an LLM is to learn patterns, relationships, grammar, semantics, and contextual meaning within
language. LLMs function by predicting the next word in a sequence based on the context of
previous words. Through this process, they can generate coherent sentences, answer questions,
summarize information,

Chunk 3:
source and use it to generate more accurate and context-aware responses. In a RAG system, when
a user submits a query, the system first converts the query into a vector representation using
embeddings. This vector is then used to search a vector database containing pre-processed
document chunks. The


In [ ]:
def retrieve_docs_with_sources(query, top_k=3):
    query_embedding = embedding_model.embed_query("query: " + query)

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    retrieved_data = []

    for match in results["matches"]:
        retrieved_data.append({
            "id": match["id"],
            "text": match["metadata"]["text"],
            "score": match["score"]
        })

    return retrieved_data


# Test
data = retrieve_docs_with_sources("What is this document about?")
print(data[0])

{'id': '5', 'text': 'source and use it to generate more accurate and context-aware responses. In a RAG system, when\na user submits a query, the system first converts the query into a vector representation using\nembeddings. This vector is then used to search a vector database containing pre-processed\ndocument chunks. The most relevant pieces of information are retrieved and passed as context to\nthe language model. The language model then generates a response based on both the user query\nand the retrieved context. This approach improves factual accuracy, reduces hallucinations, and\nallows the system to access up-to-date or domain-specific information without retraining the model.\nRAG is widely used in applications such as chatbots, document question answering systems,\nknowledge assistants, and enterprise AI solutions.\nPage 3\nLarge Language Models (LLMs) are advanced artificial intelligence systems designed to\nunderstand, process, and generate human-like language. They are buil

In [ ]:
def generate_answer_with_citations(query):

    from groq import Groq
    client = Groq(
        api_key=os.environ.get("GROQ_API_KEY")
    )

    # Step 1: Retrieve data
    retrieved = retrieve_docs_with_sources(query)

    # Step 2: Build context
    context = ""
    sources = []

    for i, item in enumerate(retrieved):
        context += f"[Source {i+1}]\n{item['text']}\n\n"
        sources.append(item)

    # Step 3: Prompt
    prompt = f"""
You are a helpful AI assistant.

Rules:
- Answer ONLY using the provided sources
- Always reference sources like [Source 1], [Source 2]
- If answer not found, say "I don't know based on the document"

Sources:
{context}

Question:
{query}

Answer:
"""

    # Step 4: Groq call
    response = client.chat.completions.create(
        model="groq/compound-mini", # Changed model name to a known groq model
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    # Step 5: Print answer + sources
    print("🤖 Answer:\n")
    print(answer)

    print("\n📚 Sources:\n")
    for i, item in enumerate(sources):
        print(f"\nSource {i+1} (Score: {item['score']:.4f}):\n")
        print(item["text"][:300])  # preview


# Test
generate_answer_with_citations("what is taj")

🤖 Answer:

I’m sorry, but none of the provided sources contain information about the Taj Mahal, so I can’t answer that question using only the given references.

📚 Sources:


Source 1 (Score: 0.7085):

an LLM is to learn patterns, relationships, grammar, semantics, and contextual meaning within
language. LLMs function by predicting the next word in a sequence based on the context of
previous words. Through this process, they can generate coherent sentences, answer questions,
summarize information,

Source 2 (Score: 0.7082):

an LLM is to learn patterns, relationships, grammar, semantics, and contextual meaning within
language. LLMs function by predicting the next word in a sequence based on the context of
previous words. Through this process, they can generate coherent sentences, answer questions,
summarize information,

Source 3 (Score: 0.7080):

an LLM is to learn patterns, relationships, grammar, semantics, and contextual meaning within
language. LLMs function by predicting the next

In [ ]:
##### chat like response

In [ ]:
def rewrite_query(query):
    global chat_history

    # Format history
    history_text = ""
    for turn in chat_history[-3:]:   # last 3 turns only
        history_text += f"User: {turn['user']}\nAssistant: {turn['assistant']}\n"

    prompt = f"""
You are a query rewriting assistant.

Convert the user's latest question into a standalone question.

Rules:
- Use chat history for context
- Make the question complete and clear
- Keep original meaning
- Do NOT answer, only rewrite

Chat History:
{history_text}

Follow-up Question:
{query}

Rewritten Question:
"""

    response = client.chat.completions.create(
        model="groq/compound-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [ ]:
def is_greeting(query):
    greetings = ["hi", "hello", "hey", "hii", "good morning", "good evening"]
    return query.lower().strip() in greetings

In [ ]:
from groq import Groq
client = Groq(
        api_key=os.environ.get("GROQ_API_KEY")
    )

In [ ]:
# Store conversation history
chat_history = []

def generate_chat_response(query):
    global chat_history

    #  Step 1: Handle greetings directly
    if is_greeting(query):
        answer = "Hello! 👋 Ask me anything about your document."

        chat_history.append({
            "user": query,
            "assistant": answer
        })
        return answer

    #  Step 2: Rewrite query
    rewritten_query = rewrite_query(query)

    # Clean bad outputs like "Rewritten Question:"
    if "Rewritten Question:" in rewritten_query:
        rewritten_query = rewritten_query.split("Rewritten Question:")[-1].strip()

    DEBUG = False  # change to True when debugging

    if DEBUG:
        print(f"\n[DEBUG] Rewritten Query: {rewritten_query}\n")

    # Step 3: Retrieve docs
    docs = retrieve_docs(rewritten_query)
    docs = docs[:3]
    context = "\n\n".join(docs)

    # Step 4: Prompt
    prompt = f"""
You are a helpful AI assistant.

Rules:
- Answer ONLY from the given context
- If answer is not in context, say "I don't know based on the document"
- Keep answer simple and clear

Context:
{context}

Question:
{rewritten_query}

Answer:
"""

    response = client.chat.completions.create(
        model="groq/compound-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    # Save history
    chat_history.append({
        "user": query,
        "assistant": answer
    })

    return answer


# Test chat loop
while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    response = generate_chat_response(user_input)
    print("Bot:", response)

You: hi
Bot: Hello! 👋 Ask me anything about your document.
You: what is rag?
Bot: RAG (Retrieval‑Augmented Generation) is a system that first retrieves relevant document chunks—by converting a user query into a vector and searching a vector database—and then feeds those retrieved pieces as context to a language model, which generates a response based on both the query and the retrieved information. This improves factual accuracy and reduces hallucinations.
You: what is llm?
Bot: An LLM (large language model) is an advanced AI system that uses deep‑learning transformer architectures, trained on massive text datasets, to understand, process, and generate human‑like language by predicting the next word in a sequence. It learns patterns, grammar, semantics, and context, and can generate coherent sentences, answer questions, summarize information, and more.
You: can you summaize above
Bot: RAG (Retrieval‑Augmented Generation) works by turning a user’s query into an embedding vector, using t